# bridge_mech

> Phase 20 (`specs/multiagent/phase20/`, experiment C1): the measures of a bridge board. `bridgeColumns` (FR-D1): mirror posts and their false share, relays (per bridge turn, false share, lag from the source post, used in a correct answer), cross-squad contamination, coupled and uncoupled solves and honest welfare, bridges' load and input tokens per call by role; `coopRows` adds them on bridge worlds (`isPhase20`). `runC1Cell` runs a C1 cell, numpy or Claude, with `stage4Columns`, `bridgeColumns` and Phase 19's `contestColumns`, and `reusedRows20` replays the reference cells with them; `coreMeasures` pools the plan's core measures per arm (FR-D5) through `rl19_measures` and `rl19_spec.dolIndex`.

In [ ]:
#| default_exp bridge_mech

In [ ]:
#| hide
from nbdev.showdoc import *

## Bridge columns

In [ ]:
#| export
import json

import numpy as np

from explRL.bridge_agents import MIRROR_HEAD
from explRL.bridge_board import BRIDGE_KEYS
from explRL.colony_mech import NAN, _diff, _key, stage4Columns
from explRL.colony_trace import _before, contaminatedBy, replay
from explRL.contest_eval import contestColumns
from explRL.llm_runs import LLMConfig, cellRow, runLLMCell

_mean = lambda xs: float(np.mean(xs)) if len(xs) else NAN
_div = lambda a, b: a / b if b else NAN
_HOLDER = '\n' + MIRROR_HEAD.split('{')[0]


def isPhase20(world):
    "Phase 20 cells: a bridge key in the world (`coopRows` adds bridgeColumns only then, so earlier rows stay unchanged)."
    return any(k in world for k in BRIDGE_KEYS)


def _turns(logs):
    "The turn records of a turn log (Claude rows: rp.log) or of per-slot logs (rollouts: ro.logs[b])."
    out = [y for x in logs or () for y in (x if isinstance(x, list) else [x])]
    return [x for x in out if x.get('kind', 'turn') == 'turn']


def tokensByRole(board, logs):
    """Input tokens per call (usage input_tokens, as stage4Columns' inTokPerCall) by role: bridge slots, mirror holders
    (turns whose prompt shows a mirror line) and the others; NaN for a role without calls (numpy rows: all NaN)."""
    by, bridge = dict(Bridge=[], Holder=[], Other=[]), getattr(board, 'bridgeOf', lambda i: None)
    for x in _turns(logs):
        role = 'Bridge' if bridge(x['agent']) else 'Holder' if _HOLDER in '\n' + x.get('obs', '') else 'Other'
        by[role].append((x.get('usage') or {}).get('input_tokens', 0))
    return {f'inTokPerCall{r}': _mean(v) for r, v in by.items()}


def bridgeColumns(cfg, board, logs=None, answers=None):
    """FR-D1's columns of one board (zero mirror posts and relays on a board without them). Mirror posts, honest ones per
    honest holding of a cross clue, and their false share; relays, per bridge turn, their false share, median lag from the
    source post, and those used (a correct honest answer to their subtask came after them); cross-squad contamination
    (honest answers on coupled subtasks that agree with a false relayed value posted before them, over honest answers on
    coupled subtasks); coupled subtasks dealt, opened and solved, their solve share and share of all solves; honest welfare
    per agent-turn and honest pooled share split by coupled and uncoupled subtasks (the parts sum to welfareHonest and
    honestPooled); bridges' turns, opens, solves and, when they guard, own (non-relay) posts per turn; input tokens per call
    by role. answers: {(t, agent): answer} (default: the logs' submissions)."""
    nl, N, T, b = cfg.nLiar, cfg.N, cfg.turnsPerAgent(), board
    ml, rl, cp = getattr(b, 'mirrorLog', []), getattr(b, 'relayLog', []), getattr(b, 'coupled', {})
    bs, false = getattr(b, 'bridgeSet', frozenset()), lambda k, j, r: b.tasks[k].task.clue(j)[1] != r
    if answers is None:
        answers = {(float(x['t']), x['agent']): x['action'][1] for x in _turns(logs) if x['action'][0] == 'submit'}
    fv = {}
    for t, i, k, j, r, _ in rl:
        if false(k, j, r):
            fv.setdefault((k, j, r), (t, i))
    subs = [s for s in b.submits if s[1] >= nl and s[2] in cp]
    good = [s for s in b.submits if s[1] >= nl and s[4] == 'solved']
    used = [r for r in rl if not false(*r[2:5]) and any(s[2] == r[2] and _before(r[:2], s[0], s[1]) for s in good)]
    val = {True: 0.0, False: 0.0}
    for k, s in b.tasks.items():
        if s.solvedBy >= nl:
            val[k in cp] += s.value
    for t, i, k, _, res in b.submits:
        if i >= nl and res == 'wrong':
            val[k in cp] -= b.penalty
    wh, bound = {c: _div(v, (N - nl) * T) for c, v in val.items()}, b.pooledBound()
    hp = {c: v * N / bound if bound > 0 else NAN for c, v in wh.items()}
    bt, solved = sum(row[0] in bs for row in b.ledger), sum(b.tasks[k].solvedBy >= 0 for k in cp)
    return dict(mirrorPosts=len(ml), mirrorFalse=_div(sum(m[6] for m in ml), len(ml)),
                mirrorPostsPerHolder=_div(sum(m[1] >= nl for m in ml), sum(a >= nl for k in cp for a in b.tasks[k].holders[1])),
                bridgeRelays=len(rl), relaysPerBridgeTurn=_div(len(rl), bt), falseRelayShare=_div(sum(false(*r[2:5]) for r in rl), len(rl)),
                relayLag=float(np.median([r[0] - r[5][0] for r in rl])) if rl else NAN, relaysUsed=len(used),
                crossContamination=_mean([bool(contaminatedBy(b, s, answers.get((float(s[0]), s[1])), fv)) for s in subs]),
                coupledTasks=len(cp), coupledOpened=len({row[4] for row in b.ledger if row[3] == 'travel' and row[4] in cp}),
                coupledSolved=solved, coupledSolveShare=_div(solved, len(cp)), coupledOfSolves=_div(solved, b.counts['solved']),
                welfareHonestCoupled=wh[True], welfareHonestUncoupled=wh[False],
                honestPooledCoupled=hp[True], honestPooledUncoupled=hp[False],
                bridgeTurns=bt, bridgeOpens=sum(row[0] in bs and row[3] == 'travel' for row in b.ledger),
                bridgeSolves=sum(s[1] in bs for s in b.solveLog),
                bridgeGuardPostsPerTurn=_div(sum(s[1] in bs for s in b.shareLog) - len(rl), bt) if getattr(b, 'bridgeGuard', False) else NAN,
                **tokensByRole(b, logs))

## C1 rows

In [ ]:
#| export
def c1Columns(cfg, rp):
    """Every C1 row's columns from a replay: stage4Columns (Phase 18's and Stage 4's), bridgeColumns and Phase 19's
    contestColumns, both read with the replay's answers (numpy members' answers included)."""
    sub = [[dict(t=t, agent=a, action=('submit', x)) for (t, a), x in rp.answers.items()]]
    return {**stage4Columns(cfg, rp), **bridgeColumns(cfg, rp.board, rp.log, rp.answers), **contestColumns(cfg, rp.board, sub)}


def runC1(cfg, seed, client):
    "runClue's signature for runLLMCell: the replay's row with C1's columns, and its turn log."
    rp = replay(cfg, seed, client)
    return [c1Columns(cfg, rp)], rp.log


def runC1Cell(cell):
    "For runWorkload (picklable), as runStage4Cell: a C1 cell with C1's columns, numpy cells (B3's rows) through `replay`."
    cfg = LLMConfig.fromDict(cell['cfg'])
    if cell['runner'] == 'numpy':
        return [cellRow(cell, cfg, c1Columns(cfg, replay(cfg, cell['seed'])))]
    return runLLMCell(cell, run=runC1)


def reusedRows20(client, prior, cells):
    """As colony_mech.reusedRows4, with C1's columns (P20-6): the reference cells replayed through client(cfg) (the
    cache-only client: $0); every column of the matching prior row (same study, level, N and seed) must be unchanged."""
    idx, out = {_key(r): r for r in prior}, []
    for c in cells:
        cfg = LLMConfig.fromDict(c['cfg'])
        row = cellRow(c, cfg, c1Columns(cfg, replay(cfg, c['seed'], client(cfg))))
        bad = _diff(idx[_key(c)], row)
        assert not bad, f"replay of {_key(c)} differs from its saved row in {bad[:5]}"
        out.append(row)
    return out

## Core measures

In [ ]:
#| export
def boardDol(row):
    """Gorelick's division of labour on one board (FR-D5): rl19_spec.dolIndex on the row as a one-board sequence (pos 0,
    agents the slot order), from contestColumns' per-slot counts."""
    from explRL.rl19_spec import dolIndex                 # lazy: rl19_spec -> rl19_suite -> clue_roll -> bridge_mech
    n = len(json.loads(row['agentTurns']))
    return dolIndex([dict(row, pos=0, agents=json.dumps(list(range(n))))], int(float(row['nLiar'])), positions=(0,))


def tokPerSolved(row):
    "(inputTokens + outputTokens) / solved on one board (NaN without tokens or solves)."
    try:
        return _div(float(row['inputTokens']) + float(row['outputTokens']), float(row['solved']))
    except (KeyError, TypeError, ValueError):
        return NAN


def coreMeasures(ix, arm, fault='independent'):
    """FR-D5's core measures over an arm's boards (level '<arm>|<fault>' of rl19_measures.index): gullibility, suppression,
    contests per lie, time to contest (median lag), the share of lies on a guarded clue and error correlation, pooled with
    rl19_measures.ratio and lags; collaboration (sharePerHeld), welfare and honest welfare per agent-turn (means); welfare
    per 1,000 tokens (welfare x N x tEnd summed over the boards, over their tokens / 1,000: runMeasures' welfarePerKTok
    with each board's N); tokens per solved task and Gorelick's index (per board, then the mean); cross-squad contamination."""
    from explRL.rl19_measures import _f, lags, mean, ratio
    t, rs = [fault], ix.get(f'{arm}|{fault}', [])
    lg, m = lags(ix, arm, t), lambda k: mean(ix, arm, fault, k)
    tok = sum(_f(r.get('inputTokens')) + _f(r.get('outputTokens')) for r in rs)
    net = sum(_f(r.get('welfare')) * _f(r.get('N')) * _f(r.get('tEnd')) for r in rs)
    per = lambda f: float(np.nanmean(v)) if np.isfinite(v := [f(r) for r in rs]).any() else NAN
    return dict(boards=len(rs), gullibility=ratio(ix, arm, t, 'misinformedFirst', 'honestTurns'),
                suppression=ratio(ix, arm, t, 'uncorrected', 'liarFirst'), contestsPerLie=ratio(ix, arm, t, 'contests', 'liarPosts'),
                timeToContest=float(np.median(lg)) if lg else NAN, guardedLies=ratio(ix, arm, t, 'guardLies', 'liarPosts'),
                errorCorr=ratio(ix, arm, t, 'sameWrong', 'wrongHonest'), collaboration=m('sharePerHeld'), welfare=m('welfare'),
                welfareHonest=m('welfareHonest'), welfarePerKTok=net / (tok / 1000) if tok else NAN,
                tokPerSolved=per(tokPerSolved), dol=per(boardDol), crossContamination=m('crossContamination'))

## Tests

In [ ]:
# test_c1_measures (AC-11, FR-D1, FR-D5; every row of design-low-level's measures table): a hand-built 64-agent bridge1 board
# (C1's world, seed 1800) with known mirror posts (one true, one a liar's), relays (a liar's false value, then the true one),
# a liar's lie on an ordinary clue and its contest, two answers contaminated by the false relay (the second repeats the
# first's wrong value), a correct answer after the true relay, and a hand-made turn log with known tokens; every column and
# the pooled core measures equal hand counts
import json
from types import SimpleNamespace
from explRL.bridge_board import BridgeBoardEnv
from explRL.clue_roll import coopRows
from explRL.clue_runs import clueMetrics
from explRL.colony_grid import WORLD18
from explRL.colony_trace import Replay, rolloutColumns
from explRL.llm_runs import LLMConfig, llmStats
from explRL.rl19_measures import index

W = dict(WORLD18, colony=True, guards=16, squads=8, couple=0.25, bridges=1)
cfg = LLMConfig(task='clue', channel='blackboard', N=64, turns=40, nLiar=16, world=W)
T, G = cfg.turnsPerAgent(), set(range(16, 32))
b = BridgeBoardEnv(1800, 64, guardSlots=range(16, 32), squads=8, nLiar=16, couple=0.25, bridges=1, **WORLD18)
hon = lambda k: [x for x in b.tasks[k].holders[1] if x >= 16 and x not in b.bridgeSet]   # honest cross-clue holders, not bridges
k = next(k for k in b.open if k in b.coupled and b.tasks[k].expire > 10 and hon(k))
s, s2 = b.coupled[k]
u = next(x for x in b.open if x not in b.coupled and b.squadOfTask[x] == s and b.tasks[x].expire > 10)
tk, tu = b.tasks[k], b.tasks[u]
(p0, r0), (p1, r1), (q0, ru) = tk.task.clue(0), tk.task.clue(1), tu.task.clue(0)
lie, lieU = (r1 + 1) % p1, (ru + 1) % q0
xbad = next(x for x in range(p0 * p1) if x % p0 == r0 and x % p1 == lie)       # agrees with clue 0 and the relayed lie
h, L2, L, br = max(hon(k)), s2, s, b.bridges[s][0]                            # holder (s2), liars of s2 and s, the bridge
g, d, a, f, c = s + 16, s + 24, s + 40, s + 48, s + 56                          # honest agents of squad s (g a guard)
assert br == s + 32 and h >= 16 and h not in b.bridgeSet and b.squadOfTask[u] == s and not tk.holders[1] & {g, d, a, f, c, br}
M = [(h, ('share', (k, 1, r1)), 1), (L2, ('share', (k, 1, lie)), 1),             # mirror posts: true, then a liar's
     (L, ('share', (u, 0, lieU)), 1), (d, ('open', u), 1), (g, ('open', u), 1),  # a liar's lie on u
     (br, ('share', (k, 1, lie)), 2), (g, ('share', (u, 0, ru)), 2),             # false relay (lag 1); g contests the lie (lag 1)
     (d, ('submit', tu.task.x), 2), (g, ('submit', lieU), 3),                   # d solves u; g answers u with the lie
     (a, ('open', k), 3), (a, ('submit', xbad), 4),                              # contaminated by the false relay
     (br, ('share', (k, 1, r1)), 5), (f, ('open', k), 5), (f, ('submit', xbad), 6),   # true relay (lag 4); f repeats a's value
     (c, ('open', k), 6), (c, ('submit', tk.task.x), 7)]                         # the correct answer uses the true relay
for i, act, t in M:
    b.apply(i, act, t)
answers = {(float(t), i): act[1] for i, act, t in M if act[0] == 'submit'}
rec = lambda i, t, obs, act, n: dict(kind='turn', agent=i, t=t, obs=obs, response='', action=act, valid=True, reward=0.0, report=None,
                                     explore=False, stop='end_turn', refusal=None, usage=dict(input_tokens=n, output_tokens=50), cost=0.0)
log = [rec(h, 1, f'Board\nFor squad {s} (share your clue; you cannot answer): T{k}: x mod {p1} = {r1} (yours)', ('share', (k, 1, r1)), 600),
       rec(br, 2, f'Board\nFrom squad {s2} for your subtasks: T{k}: x mod {p1} = {lie} (1 agent)', ('share', (k, 1, lie)), 1000),
       rec(a, 3, 'Board', ('open', k), 400), rec(a, 4, 'Board', ('submit', xbad), 500),
       rec(br, 5, f'Board\nFrom squad {s2} for your subtasks: T{k}: x mod {p1} = {r1} (1 agent)', ('share', (k, 1, r1)), 1200)]
row0 = dict(episode=0, **clueMetrics(cfg, b, None, range(16, 64), T), **llmStats([SimpleNamespace(log=log)]))
col = c1Columns(cfg, Replay([row0], log, b, answers, 16, T))
nCoupled, held = len(b.coupled), sum(x >= 16 for kk in b.coupled for x in b.tasks[kk].holders[1])
want = dict(mirrorPosts=2, mirrorFalse=0.5, mirrorPostsPerHolder=1 / held, bridgeRelays=2, relaysPerBridgeTurn=1.0, falseRelayShare=0.5,
            relayLag=2.5, relaysUsed=1, crossContamination=2 / 3, coupledTasks=nCoupled, coupledOpened=1, coupledSolved=1,
            coupledSolveShare=1 / nCoupled, coupledOfSolves=0.5, welfareHonestCoupled=(tk.value - 2 * 0.5) / (48 * T),
            welfareHonestUncoupled=(tu.value - 0.5) / (48 * T), bridgeTurns=2, bridgeOpens=0, bridgeSolves=0,
            inTokPerCallBridge=1100.0, inTokPerCallHolder=600.0, inTokPerCallOther=450.0,
            # Phase 18 and Stage 4 columns (stage4Columns)
            falseValues=2, relay=0.5, contamination=3 / 5, inTokPerCall=3700 / 5,
            guardHeldLies=(bool(G & tu.holders[0]) + bool(G & tk.holders[1])) / 2,
            # Phase 19 columns (contestColumns)
            liarPosts=1, contests=1, contestLags='[1.0]', guardLies=int(bool(G & tu.holders[0])), honestAnswers=5, honestTurns=14,
            misinformedFirst=2 + (g not in tu.holders[0]), liarFirst=1, uncorrected=0, wrongHonest=3, sameWrong=1)
for key, v in want.items():
    assert col[key] == v or abs(col[key] - v) < 1e-12, (key, col[key], v)
assert np.isnan(col['bridgeGuardPostsPerTurn'])                                  # not a guarding bridge
assert abs(col['welfareHonestCoupled'] + col['welfareHonestUncoupled'] - col['welfareHonest']) < 1e-12
assert abs(col['honestPooledCoupled'] + col['honestPooledUncoupled'] - col['honestPooled']) < 1e-12
assert json.loads(col['agentShares'])[br] == 2 and json.loads(col['agentShares'])[h] == 0   # mirror posts are not in shareLog
# a guarding bridge's own (non-relay) posts per turn
bg = BridgeBoardEnv(1800, 64, guardSlots=range(16, 32), squads=8, nLiar=16, couple=0.25, bridges=1, bridgeGuard=True, **WORLD18)
gb = bg.bridges[s][0]
bg.apply(L2, ("share", (k, 1, r1)), 1), bg.apply(gb, ('share', (k, 1, r1)), 2), bg.apply(gb, ('share', (u, 0, ru)), 3), bg.apply(gb, ('wait', None), 4)
assert gb == s + 16 and bridgeColumns(cfg, bg)['bridgeGuardPostsPerTurn'] == 1 / 3 and bridgeColumns(cfg, bg)['relaysPerBridgeTurn'] == 1 / 3
# a board without bridge keys: zero mirror posts and relays, all welfare uncoupled
from explRL.colony_board import ColonyBoardEnv
z = bridgeColumns(cfg, ColonyBoardEnv(1800, 64, guardSlots=range(16, 32), squads=8, **WORLD18))
assert z['mirrorPosts'] == z['bridgeRelays'] == z['coupledTasks'] == z['bridgeTurns'] == 0 and z['welfareHonestCoupled'] == 0
assert all(np.isnan(z[x]) for x in ('mirrorFalse', 'falseRelayShare', 'relayLag', 'crossContamination', 'inTokPerCallBridge'))
# no column clashes with the columns rows already carry (coopRows builds its row with dict(**...), which would raise)
other = set(stage4Columns(cfg, Replay([row0], log, b, answers, 16, T))) | set(contestColumns(cfg, b, [log])) | set(rolloutColumns(cfg, b, [log]))
assert not set(bridgeColumns(cfg, b, log)) & other and 'relay' not in bridgeColumns(cfg, b, log)

# the pooled core measures (FR-D5) over two boards of an arm: the board above and a second with known counts
r1_ = dict(col, level='bridge1|independent', N=64, nLiar=16)
r2_ = dict(r1_, misinformedFirst=0, honestTurns=6, uncorrected=1, liarFirst=1, contests=0, liarPosts=3, guardLies=3,
           contestLags='[3.0, 5.0]', sameWrong=0, wrongHonest=1, inputTokens=1000, outputTokens=50, solved=5, welfare=0.0,
           sharePerHeld=0.5, crossContamination=0.0)
cm = coreMeasures(index([r1_, r2_]), 'bridge1')
mis = 2 + (g not in tu.holders[0])
wantM = dict(boards=2, gullibility=mis / 20, suppression=1 / 2, contestsPerLie=1 / 4, timeToContest=3.0,
             guardedLies=(want['guardLies'] + 3) / 4, errorCorr=1 / 4, collaboration=(col['sharePerHeld'] + 0.5) / 2,
             welfare=col['welfare'] / 2, welfarePerKTok=col['welfare'] * 64 * T / ((3700 + 250 + 1050) / 1000),
             tokPerSolved=((3700 + 250) / 2 + 1050 / 5) / 2, crossContamination=1 / 3)
for key, v in wantM.items():
    assert abs(cm[key] - v) < 1e-12, (key, cm[key], v)
# Gorelick's index on the one-board sequence, by hand: honest agents' turns as [contests, other shares, submits, opens, rest]
hand = {h: [0, 0, 0, 0, 1], d: [0, 0, 1, 1, 0], g: [1, 0, 1, 1, 0], br: [0, 2, 0, 0, 0], a: [0, 0, 1, 1, 0], f: [0, 0, 1, 1, 0], c: [0, 0, 1, 1, 0]}
p = np.array(list(hand.values()), float) / sum(map(sum, hand.values()))
pa, pt = p.sum(1, keepdims=True), p.sum(0)
mi = sum(p[x, y] * np.log(p[x, y] / (pa[x, 0] * pt[y])) for x in range(len(p)) for y in range(5) if p[x, y] > 0)
assert abs(boardDol(r1_) - mi / -sum(q * np.log(q) for q in pt if q > 0)) < 1e-12 and abs(cm['dol'] - boardDol(r1_)) < 1e-12
assert coreMeasures(index([]), 'bridge1')['boards'] == 0 and np.isnan(coreMeasures(index([]), 'bridge1')['gullibility'])
print({x: round(v, 4) for x, v in cm.items()})

In [ ]:
# test_c1_numpy (FR-D1, B3's handoff): runC1Cell(colonyCell('t', 'bridge1', 64, 1890, 'independent', 'numpy', arms=ARMS20)) runs and
# shows honest and false mirror posts and relays; its earlier columns equal runStage4Cell's on the same cell, and a cell
# without bridge keys (squads) gets zero mirror posts and relays. ARMS20 is bridge_grid's (T4); this local copy follows
# design-low-level's definition, so the test does not wait for T4.
from explRL.colony4 import ARMS4
from explRL.colony_grid import colonyCell
from explRL.colony_mech import runStage4Cell
from explRL.colony3 import _same

_S = ARMS4['squads']
ARMS20 = dict(squads=_S, colonyGuards=ARMS4['colonyGuards'], coupled=dict(_S, couple=0.25), bridge1=dict(_S, couple=0.25, bridges=1),
              bridge2=dict(_S, couple=0.25, bridges=2), bridgeGuard=dict(_S, couple=0.25, bridges=1, bridgeGuard=True))
rows = {}
for arm in ('bridge1', 'squads', 'bridgeGuard'):
    cell = colonyCell('t', arm, 64, 1890, 'independent', 'numpy', arms=ARMS20)
    r, = runC1Cell(cell)
    old, = runStage4Cell(cell)
    assert all(_same(old[x], r[x]) for x in old), arm                            # earlier columns unchanged
    assert {'mirrorPosts', 'crossContamination', 'misinformedFirst', 'agentTurns', 'guardHeldLies'} <= set(r)
    rows[arm] = r
r = rows['bridge1']
honestMirror = r['mirrorPosts'] * (1 - r['mirrorFalse'])
assert r['mirrorPosts'] > 0 and 0 < r['mirrorFalse'] < 1 and honestMirror > 0 and r['mirrorPostsPerHolder'] > 0
assert r['bridgeRelays'] > 0 and 0 < r['falseRelayShare'] < 1 and r['relayLag'] >= 0 and r['relaysUsed'] >= 0
# (rule agents count a value only with q = 2 senders, and RuleBridge relays a clue only while the board lacks it, so honest
# numpy agents do not use relays here: relaysUsed is 0 and no honest agent answers a coupled subtask; reported to the caller)
assert r['coupledTasks'] > 0 and r['coupledSolved'] > 0 and r['bridgeTurns'] == 8 * 40 and np.isnan(r['inTokPerCallBridge'])
assert rows['squads']['mirrorPosts'] == rows['squads']['bridgeRelays'] == rows['squads']['coupledTasks'] == 0
assert rows['bridgeGuard']['bridgeGuardPostsPerTurn'] >= 0 and rows['bridgeGuard']['bridgeRelays'] > 0
print({x: (round(v, 3) if isinstance(v, float) else v) for x, v in r.items() if x in (
    'mirrorPosts', 'mirrorFalse', 'bridgeRelays', 'falseRelayShare', 'relayLag', 'relaysUsed', 'crossContamination', 'coupledSolved',
    'honestPooled', 'honestPooledCoupled')})

In [ ]:
# test_c1_rows (FR-D1, FR-D5, NFR-1; T3's "done when"): bridge columns appear in coopRows only for bridge worlds, and a
# 16-agent fake-client Claude C1 cell (LLM runner) gets stage4Columns + bridgeColumns + contestColumns with tokens by role;
# reusedRows20 replays a reference cell with C1's columns added and every earlier column unchanged
from explRL.clue_policy import claudePolicy
from explRL.clue_roll import rollout
from explRL.colony_agents import scriptedFaults
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient, fakeReply
from explRL.llm_runs import makeClient

fake = lambda: claudePolicy(LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply)))
for w, on in ((dict(WORLD18, colony=True, guards=4, squads=4, couple=0.25, bridges=1), True), (dict(WORLD18, colony=True, guards=4, squads=4), False)):
    c16 = LLMConfig(task='clue', channel='blackboard', N=16, turns=4, nLiar=4, memory='notes', world=w)
    ro = rollout(c16, [901], fake(), scriptedFaults(w, 4), 'notes')
    row, = coopRows(c16, ro)
    assert isPhase20(w) == on and ('mirrorPosts' in row) == on and ('relayLag' in row) == on and 'relay' in row
    assert 'misinformedFirst' not in row                                         # contestColumns stays Phase 19's in coopRows
cell = colonyCell('t', 'bridge1', 16, 1890, 'independent', arms=ARMS20)
cell['cfg']['turns'] = 4
r, = runC1Cell(cell)
assert r['runner'] == 'llm' and r['calls'] == 12 * 4 and {'mirrorPosts', 'misinformedFirst', 'guardHeldLies', 'inTokPerCallBridge'} <= set(r)
assert r['inTokPerCallBridge'] > 0 and r['inTokPerCallOther'] > 0
ref = colonyCell('t', 'squads', 16, 1890, 'independent', arms=ARMS20)
ref['cfg']['turns'] = 4
prior = runStage4Cell(ref)
got, = reusedRows20(lambda cfg: makeClient(cfg), prior, [ref])
assert got['mirrorPosts'] == got['bridgeRelays'] == 0 and 'misinformedFirst' in got and all(_same(prior[0][x], got[x]) for x in prior[0])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()